# TimeSformer

Bertasius, G., Wang, H., & Torresani, L. (2021).  
*Is Space-Time Attention All You Need for Video Understanding?*

## Overview

- TimeSformer adapts the standard Transformer architecture to video by enabling spatiotemporal feature learning directly from sequences of frame-level patches.

- The model extends Vision Transformer (ViT) to video by applying self-attention over space-time representations.

- Videos are treated as sequences of patches extracted from individual frames. Each patch is linearly mapped into an embedding and augmented with positional information.

- The self-attention formulation remains the same as the original Transformer:

$$
\mathrm{Attention}(Q,K,V)
=
\mathrm{softmax}
\left(
\frac{QK^\top}{\sqrt{D_h}}
\right)V
$$

- The main difference is that tokens now represent spatiotemporal patches instead of words or image-only patches.

- Standard self-attention is computationally expensive for video because similarities must be computed between all token pairs. For example, if a video contains $F=8$ frames and each frame contains $N=196$ patches, the Transformer processes $NF=1568$ tokens, requiring attention over:

    $$
    1568^2 \approx 2.46 \text{ million}
    $$

    query-key comparisons within a single attention layer.

- The paper proposes several scalable space-time attention schemes. Among them, the divided attention architecture, where temporal attention and spatial attention are separately applied within each block, achieves the best video classification performance.

- Compared to CNNs, Transformers impose fewer restrictive inductive biases and can directly model both local and long-range dependencies through self-attention.

- Unlike convolutional kernels, which are inherently local, self-attention can compare features across arbitrary space-time locations.

- Compared to 3D convolutional networks, TimeSformer trains faster, can achieve higher inference efficiency, and can process much longer video clips.

- The architecture remains largely unchanged from the original Transformer design, with the main extension being the application of self-attention over space-time video tokens.

## Input Representation

TimeSformer extends the Vision Transformer (ViT) formulation from images to videos.

Instead of processing a single image divided into patches, TimeSformer processes a video clip composed of multiple frames.

Given:
- $F$: number of frames
- $H \times W$: frame resolution
- $P \times P$: patch size

Each frame is divided into:
$$
N = \frac{HW}{P^2}
$$
non-overlapping patches.

Each patch is represented as:
$$
x_{(p,t)}
$$
where:
- $p$: spatial patch index
- $t$: temporal frame index

Thus, tokens are now indexed by both space and time.

---

# Linear Embedding

Each patch is flattened and projected into an embedding space:

$$
z^{(0)}_{(p,t)} = E x_{(p,t)} + e^{pos}_{(p,t)}
$$

where:
- $E$: learnable linear projection matrix
- $e^{pos}_{(p,t)}$: spatiotemporal positional embedding

The resulting sequence of embeddings becomes the Transformer input.

---

# CLS Token

A special learnable classification token is added:

$$
z^{(0)}_{(0,0)}
$$

The index $(0,0)$ is reserved exclusively for the CLS token because:
- real patches start from:
  $$
  p = 1,\dots,N
  $$
- real frames start from:
  $$
  t = 1,\dots,F
  $$

The CLS token acts as a global information aggregator for video classification.

---

# Query-Key-Value Computation

For each Transformer layer $\ell$ and attention head $a$:

$$
q^{(\ell,a)}_{(p,t)} = W_Q^{(\ell,a)} LN(z^{(\ell-1)}_{(p,t)})
$$

$$
k^{(\ell,a)}_{(p,t)} = W_K^{(\ell,a)} LN(z^{(\ell-1)}_{(p,t)})
$$

$$
v^{(\ell,a)}_{(p,t)} = W_V^{(\ell,a)} LN(z^{(\ell-1)}_{(p,t)})
$$

where:
- $\ell$: Transformer layer index
- $a$: attention head index
- $LN$: LayerNorm

This is identical to the original Transformer formulation.

The only difference is that tokens now represent spatiotemporal patches instead of words or image patches.

---

# Self-Attention Computation

For a query token:
$$
(p,t)
$$

attention is computed against all candidate tokens:
$$
(p', t')
$$

The attention weights are:

$$
\alpha_{(p,t)}
=
\mathrm{softmax}
\left(
\frac{
q_{(p,t)}^\top k_{(p',t')}
}{
\sqrt{D_h}
}
\right)
$$

where:
- $p'$: candidate spatial patch index
- $t'$: candidate temporal frame index

This means:
- the current token compares itself with patches from all frames
- attention is fully spatiotemporal

---

# Relation to Original Transformer

Original Transformer attention:

$$
\alpha_{ij}
=
\mathrm{softmax}
\left(
\frac{
q_i^\top k_j
}{
\sqrt{d_k}
}
\right)
$$

TimeSformer attention:

$$
\alpha_{(p,t),(p',t')}
=
\mathrm{softmax}
\left(
\frac{
q_{(p,t)}^\top k_{(p',t')}
}{
\sqrt{D_h}
}
\right)
$$

Thus:
- the mathematical formulation is unchanged
- only the token indexing changes

The original token index:
$$
i
$$
is replaced by:
$$
(p,t)
$$

---

# Matrix Formulation

The formulation remains:

$$
\mathrm{Attention}(Q,K,V)
=
\mathrm{softmax}
\left(
\frac{QK^\top}{\sqrt{D_h}}
\right)V
$$

TimeSformer simply interprets rows and columns as spatiotemporal tokens.

---

# Spatial-Only Attention

To reduce computational cost, TimeSformer restricts attention to a single dimension.

For spatial attention:
- temporal index is fixed:
  $$
  t' = t
  $$

The token:
$$
(p,t)
$$
attends only to:
$$
(p', t)
$$

This means:
- attention occurs only inside the same frame
- spatial relationships are modeled independently per frame

The attention computation itself remains unchanged.

Only the allowed token interactions are restricted.

---

# Temporal-Only Attention

For temporal attention:
- spatial index is fixed:
  $$
  p' = p
  $$

The token:
$$
(p,t)
$$
attends only to:
$$
(p, t')
$$

This models temporal evolution of the same spatial location across frames.

---

# Key Insight

TimeSformer does not introduce a new attention operation.

The core attention formulation remains identical to the original Transformer.

The main innovation is:
- applying attention over spatiotemporal tokens
- restricting interactions to spatial or temporal dimensions to reduce computational complexity
- factorizing video attention into more efficient components

# Encoding in TimeSformer

After computing the attention weights, the Transformer updates each token representation by combining information from the value vectors.

For a token indexed by:
$$
(p,t)
$$

the output of attention for layer $\ell$ and attention head $a$ is:

$$
s^{(\ell,a)}_{(p,t)}
=
\alpha^{(\ell,a)}_{(p,t),(0,0)}v^{(\ell,a)}_{(0,0)}
+
\sum_{p'=1}^{N}
\sum_{t'=1}^{F}
\alpha^{(\ell,a)}_{(p,t),(p',t')}
v^{(\ell,a)}_{(p',t')}
$$

where:
- $\alpha$: attention weights
- $v$: value vectors
- $(0,0)$: CLS token
- $(p', t')$: candidate spatial and temporal tokens

This equation represents a weighted sum of all value vectors.

Tokens receiving higher attention weights contribute more strongly to the updated representation.

The first term corresponds to the contribution of the CLS token, while the second term aggregates information from all spatiotemporal patches.

This operation is equivalent to the standard Transformer attention formulation:

$$
\mathrm{softmax}
\left(
\frac{QK^\top}{\sqrt{D_h}}
\right)V
$$

The paper expands the equation explicitly using space-time indices.

---

# Multi-Head Attention Aggregation

Each attention head produces an output vector:

$$
s^{(\ell,1)}_{(p,t)},
\dots,
s^{(\ell,A)}_{(p,t)}
$$

The outputs from all heads are concatenated and projected:

$$
z'^{(\ell)}_{(p,t)}
=
W_O
\begin{bmatrix}
s^{(\ell,1)}_{(p,t)} \\
\vdots \\
s^{(\ell,A)}_{(p,t)}
\end{bmatrix}
+
z^{(\ell-1)}_{(p,t)}
$$

where:
- $W_O$: output projection matrix
- $A$: number of attention heads

The addition of:
$$
z^{(\ell-1)}_{(p,t)}
$$
is a residual connection.

Residual connections help stabilize training and improve gradient flow in deep Transformer architectures.

---

# MLP Block

After attention, the updated representation is processed by a feed-forward network:

$$
z^{(\ell)}_{(p,t)}
=
MLP
\left(
LN
\left(
z'^{(\ell)}_{(p,t)}
\right)
\right)
+
z'^{(\ell)}_{(p,t)}
$$

where:
- $LN$: LayerNorm
- $MLP$: feed-forward multilayer perceptron

Another residual connection is applied after the MLP block.

---

# Transformer Block Flow

The overall Transformer block follows the standard Transformer pipeline:

1. LayerNorm
2. Query-Key-Value projection
3. Self-attention computation
4. Weighted aggregation of value vectors
5. Multi-head concatenation
6. Linear projection
7. Residual connection
8. LayerNorm
9. MLP
10. Residual connection

TimeSformer preserves the original Transformer architecture, with the main difference being that tokens represent spatiotemporal video patches instead of words or image-only patches.

# Classification Embedding

After the final Transformer block, the video representation is obtained from the CLS token:

$$
y = LN
\left(
z^{(L)}_{(0,0)}
\right)
\in \mathbb{R}^{D}
$$

where:
- $L$: final Transformer layer
- $(0,0)$: classification token (CLS token)
- $LN$: LayerNorm

The CLS token aggregates information from all spatiotemporal tokens throughout the Transformer layers, producing a global representation of the video clip.

The resulting embedding:
$$
y
$$
is used as the final clip-level representation.

A 1-hidden-layer MLP is appended on top of this embedding to predict the final video classes.

## Space-Time Self-Attention Models.

To improve accuracy and computational efficiency, TimeSformer uses a "Divided Space-Time Attention". That is, instead of performing spatial and temporal attention in one step, the process is divided into two steps: first, temporal attention is applied, followed by spatial attention.

**Step 1 - Temporal Attention** 

The spatial position is fixed, so the variation is along the temporal axis: $(p, t')$

Since time varies, this models movement, temporal changes, and temporal evolution. For example, if a patch covers a hand, then temporal attention can be related to movement, such as moving a hand up or down.

**Step 2 - Spatial Attention**

Now time is fixed, so the variation is along the spatial axis: $(p', t)$

Since space varies, this models visual context because relations between patches are captured.

**Difference**

**Joint attention**

```text
(p,t)
  ↓
todos los patches de todos los frames
```

**Divided attention**

```text
(p,t)
  ↓
mismo patch en distintos frames
(temporal)
```

luego

```text
(p,t)
  ↓
otros patches del mismo frame
(spatial)
```

Everything that follows (*softmax*) remains the same.

**Remarks**

In practice, joint space-time attention causes a GPU memory overflow once the spatial frame resolution reaches 448 pixels, or once the number of frames is increased to 32 and thus it is effectively not applicable to large frames or long videos. Thus, despite a larger number of parameters, divided space-time attention is more efficient than joint space-time attention when operating on higher spatial resolution, or longer videos.

## Comparison to 3D CNNs

- TimeSformer pretrained on ImageNet trains much faster on video datasets than 3D CNNs like SlowFast or I3D.

- TimeSformer achieved better accuracy on Kinetics-400 while requiring significantly fewer GPU hours.

- SlowFast and I3D require very long training schedules to reach high performance, even when pretrained on ImageNet.

- Under similar computational budgets, TimeSformer maintains higher accuracy than SlowFast and I3D.

- One advantage of TimeSformer is that it becomes more accessible for labs without massive GPU resources.

- Training TimeSformer from scratch is difficult due to the large number of parameters.

- Without ImageNet pretraining, TimeSformer can still train, but accuracy drops considerably (64.8%).

- Because of this, the paper continues using ImageNet pretraining for all experiments.

- SlowFast can be trained from scratch, but at a much higher computational cost.

- The paper compares ImageNet-1K and ImageNet-21K pretraining.

- ImageNet-21K pretraining consistently improves performance on Kinetics-400.

- On Something-Something-V2, ImageNet-1K and ImageNet-21K achieve similar accuracy.

- The paper suggests this happens because Kinetics-400 depends more on spatial scene information, which benefits from larger image pretraining datasets.

- Something-Something-V2 depends more on complex spatiotemporal reasoning, so larger image pretraining provides less advantage.

- The paper evaluates multiple TimeSformer variants:
  - standard version
  - high-resolution version
  - long-range temporal version

- Longer clips and higher resolutions can further improve performance.

- TimeSformer scales well to higher spatial resolutions and **longer video clips compared to most 3D CNNs**.

- Increasing spatial resolution improves performance up to a certain point.

- Increasing the number of video frames consistently improves accuracy.

- Due to GPU memory limitations, the paper evaluates clips up to 96 frames.

- Processing 96-frame clips is already significantly longer than typical 3D CNN inputs, which usually process only 8–32 frames.

- The paper studies the importance of positional embeddings in TimeSformer.

- Three variants are compared:
  - no positional embedding
  - spatial-only positional embedding
  - spatiotemporal positional embedding

- Spatiotemporal positional embeddings achieve the best accuracy on both Kinetics-400 and Something-Something-V2.

- Spatial-only positional embeddings still perform well on Kinetics-400.

- Spatial-only positional embeddings perform much worse on Something-Something-V2.

- The paper suggests this happens because Kinetics-400 is more spatially biased, while Something-Something-V2 requires stronger temporal reasoning.
